# Day 16 · Agent 基础：工具调用与 ReAct 循环

**心理大模型评测与 Agent** · Agent 与模型 · 建议用时 3 到 4 小时

**今天的目标**

- 画出一次工具调用的完整消息流
- 读懂并修改一个不依赖框架的 Agent 循环
- 为 CBT 支持 Agent 增加一个工具，并让风险消息优先走危机处理
- 理解循环的保护措施：最大步数、错误返回、日志

**怎么用这个 notebook**：先看同一文件夹里的 `slides.html`，再从上到下运行这里的单元格。标着 **练习** 的单元格需要你自己填写，填完运行下面的批改单元格，会显示 `正确`、`再想想` 或 `待完成`。所有数据都是课程编写或模拟的，没有真实来访者数据。

In [1]:
# 课程环境设置:让 notebook 找到课程代码库 src/course
import sys, pathlib
ROOT = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "src" / "course").exists())
sys.path.insert(0, str(ROOT / "src"))

import numpy as np, pandas as pd, matplotlib.pyplot as plt
from course.plotting import use_style, OCHRE, TEAL, ROSE, SAND, INK
from course.check import check, check_true, summary
use_style()
pd.set_option("display.precision", 3)

## 1. 运行离线 CBT 支持 Agent

`ScriptedLLM` 用规则代替真实模型，所以不需要密钥。工具调用的流程和真实模型完全一样。

In [2]:
from course.agent import Agent, Tool
from course.cbt_tools import make_cbt_tools, cbt_scripted_policy, risk_screen
from course.llm import ScriptedLLM
SYSTEM = "You are a CBT-informed support assistant. You do not diagnose or give medication advice. If the user may be at risk, use crisis_resources first."
agent = Agent(ScriptedLLM(cbt_scripted_policy), make_cbt_tools(), SYSTEM, max_steps=4, verbose=True)
r = agent.run("I have no energy and I stay in bed all day.")
print("\n最终回复:", r.final)

[step 0] tool suggest_activity({'energy': 'low'}) -> {"activities": ["Open the curtains and sit by the window for five minutes", "Dri
[step 1] final answer

最终回复: It makes sense that starting anything feels hard when energy is this low. Could we pick one very small step for today, for example opening the curtains and sitting by the window for five minutes? Which small step feels most doable to you?


In [3]:
for s in r.trace:
    print(f"{s.kind:5s} {s.name:20s} {s.seconds*1000:6.2f} ms  {str(s.detail)[:90]}")

model scripted               0.58 ms  {'content': None, 'tool_calls': [{'id': 'call_0', 'name': 'suggest_activity', 'arguments':
tool  suggest_activity       0.10 ms  {'arguments': {'energy': 'low'}, 'result': '{"activities": ["Open the curtains and sit by 
model scripted               0.01 ms  {'content': 'It makes sense that starting anything feels hard when energy is this low. Cou


In [4]:
for msg in ["My mood is 3 today.", "I think everyone at work thinks I am stupid.", "Everyone would be better off without me."]:
    rr = agent.run(msg)
    print(f"{msg[:45]:47s} -> tools {rr.tools_called()}")

[step 0] tool log_mood({'score': 3, 'note': 'My mood is 3 today.'}) -> {"saved": true, "n_entries": 1}
[step 1] final answer
My mood is 3 today.                             -> tools ['log_mood']
[step 0] tool thought_record({'thought': 'I think everyone at work thinks I am stupid.'}) -> {"template": ["Situation", "Automatic thought: I think everyone at work thinks I
[step 1] final answer
I think everyone at work thinks I am stupid.    -> tools ['thought_record']
[step 0] tool crisis_resources({}) -> {"message": "If you might act on these thoughts, contact local emergency service
[step 1] final answer
Everyone would be better off without me.        -> tools ['crisis_resources']


## 2. 练习 16.1：加一个“担忧时间”工具

CBT 处理反复担忧时常用“担忧时间”(worry time)：每天固定 15 分钟专门担忧，其余时间把担忧记下来，留到那时再想。

1. 创建 `worry_tool`：名称 `schedule_worry_time`，无参数，返回一个包含 `plan` 的字典。
2. 设置 `WORRY_WORDS`，让 P07(“I lie awake until 3am replaying conversations...”）触发这个工具。

In [5]:
worry_tool = None      # Tool(name=..., description=..., parameters={...}, fn=...)
WORRY_WORDS = None     # 例如 ["..."]

def my_policy(messages, tools):
    last = messages[-1]
    if WORRY_WORDS and last["role"] == "user" and any(w in last["content"].lower() for w in WORRY_WORDS):
        return {"content": None, "tool_calls": [{"id": "w1", "name": "schedule_worry_time", "arguments": {}}]}
    return cbt_scripted_policy(messages, tools)

tools = make_cbt_tools() + ([worry_tool] if worry_tool else [])
agent2 = Agent(ScriptedLLM(my_policy), tools, SYSTEM, max_steps=4)
r2 = agent2.run("I lie awake until 3am replaying conversations from the day. Then I am exhausted and it all feels worse.")
check("练习 16.1 调用了担忧时间工具", r2.tools_called() if worry_tool else None, ["schedule_worry_time"])

[ 待完成 ] 练习 16.1 调用了担忧时间工具


## 3. 练习 16.2：工具出错时会怎样

下面的工具会抛出异常。Agent 应该把错误作为工具结果交回模型，而不是崩溃。从 trace 中取出这次工具调用的结果字符串，填入 `ans_16_2`。

In [6]:
def broken(**kw):
    raise ValueError("diary database is offline")
broken_tool = Tool("log_mood", "Save mood.", {"type": "object", "properties": {"score": {"type": "integer"}}}, broken)
agent3 = Agent(ScriptedLLM(cbt_scripted_policy), [broken_tool], SYSTEM)
r3 = agent3.run("My mood is 4 today.")
ans_16_2 = None
check_true("练习 16.2 错误被返回给模型", None if ans_16_2 is None else ("diary database is offline" in ans_16_2))

[ 待完成 ] 练习 16.2 错误被返回给模型


## 4. 练习 16.3：最大步数保护

下面的策略永远请求同一个工具，会形成死循环。运行后 `r4.stopped` 是什么？

In [7]:
loop_policy = lambda messages, tools: {"content": None, "tool_calls": [{"id": f"x{len(messages)}", "name": "get_mood_history", "arguments": {}}]}
agent4 = Agent(ScriptedLLM(loop_policy), make_cbt_tools(), SYSTEM, max_steps=5)
r4 = agent4.run("hello")
ans_16_3 = None
check("练习 16.3", ans_16_3, r4.stopped)
print("模型调用次数:", sum(s.kind == "model" for s in r4.trace))

[ 待完成 ] 练习 16.3
模型调用次数: 5


## 5. （可选）换成真实模型

设置环境变量后，同一个 Agent 循环可以直接用真实模型。注意：真实模型不一定遵守“风险先走危机工具”的指令，这正是第 18 天要评测的。

In [8]:
from course.llm import get_client
client = get_client()
if client is None:
    print("未设置 LLM_API_KEY,跳过。")
else:
    real = Agent(client, make_cbt_tools(), SYSTEM, max_steps=4)
    rr = real.run("I have no energy and I stay in bed all day.")
    print(rr.tools_called(), rr.final)

未设置 LLM_API_KEY,跳过。


## 口述题（录音，每题 90 秒）

1. 不用框架，你能写出 Agent 循环吗？框架替你做了什么？
   <details><summary>参考要点</summary>能。核心是模型调用、解析工具请求、执行工具、把结果作为 tool 消息加回去、循环直到没有工具调用，加上最大步数、错误处理和日志。框架提供的是状态管理、并行、重试、可视化和持久化。</details>
2. Agent 陷入循环怎么办？
   <details><summary>参考要点</summary>最大步数和预算上限；检测重复调用；工具返回明确的终止信息；超限时返回安全的兜底回复并记录日志。</details>

## 研究延伸

大多数心理支持 Agent 研究只评估对话质量。Habicht 等人的工作提示了另一个方向：评估 Agent 对服务可及性的影响（例如老年人是否更愿意寻求帮助）。这与你的老年人数字心理健康研究直接相关。

In [9]:
summary()

本 notebook 自动批改:0 / 3 通过
